In [1]:
import json
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. MEMORY-EFFICIENT GRAPH CONSTRUCTION
def build_graph_full(json_path):
    print("Loading full dataset... (Requires 16GB+ System RAM)")
    
    user_ids, friends_list, features_list, labels = [], [], [], []
    
    # Parse line-by-line to save memory
    with open(json_path, 'r', encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            user_ids.append(row['user_id'])
            friends_list.append(row.get('friends', ''))
            
            # Extract features
            features_list.append([
                row.get('review_count', 0),
                row.get('useful', 0),
                row.get('funny', 0),
                row.get('cool', 0),
                row.get('fans', 0),
                row.get('average_stars', 0.0)
            ])
            
            # Extract label (1 if Elite, 0 if not)
            elite = row.get('elite', '')
            labels.append(1 if len(elite) > 0 and elite != 'None' else 0)

    print(f"Loaded {len(user_ids)} users. Building network topology...")
    
    # Create O(1) lookup dictionary for user IDs
    user_id_map = {uid: i for i, uid in enumerate(user_ids)}
    
    # Build Edges
    src, dst = [], []
    for u_idx, friends_str in enumerate(friends_list):
        if friends_str == 'None' or not friends_str:
            continue
        friends = [f.strip() for f in friends_str.split(',')]
        for friend in friends:
            if friend in user_id_map:
                v_idx = user_id_map[friend]
                src.append(u_idx)
                dst.append(v_idx)
                
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    
    # Scale Features
    print("Scaling features...")
    scaler = StandardScaler()
    x = torch.tensor(scaler.fit_transform(features_list), dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.long)
    
    # Compile PyG Data Object
    data = Data(x=x, edge_index=edge_index, y=y)
    
    # 9:1 Train/Test Split
    indices = np.arange(len(user_ids))
    train_idx, test_idx = train_test_split(indices, test_size=0.1, random_state=42)
    
    train_mask = torch.zeros(len(user_ids), dtype=torch.bool)
    test_mask = torch.zeros(len(user_ids), dtype=torch.bool)
    train_mask[train_idx] = True
    test_mask[test_idx] = True
    
    data.train_mask = train_mask
    data.test_mask = test_mask
    
    print(f"Graph Built: {data.num_nodes} nodes, {data.num_edges} edges.")
    return data

# 2. GAT MODEL DEFINITION
class GATNet(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super(GATNet, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# 3. TRAINING ROUTINE WITH NEIGHBOR LOADER
if __name__ == '__main__':
    # 1. Load Data
    data = build_graph_full('yelp_dataset/yelp_academic_dataset_user.json') 
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = GATNet(in_channels=data.num_node_features, hidden_channels=16, out_channels=2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

    # 2. Transfer full graph to device for Full-Batch Training
    # (Memory footprint is small enough since features are only 6 dimensions)
    data = data.to(device)

    # 3. Train Loop
    print("Beginning Full-Batch Training...")
    model.train()
    epochs = 200 # More epochs needed for full-batch gradient descent
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        out = model(data.x, data.edge_index)
        
        # Compute loss only on training nodes
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:>3} | Loss: {loss.item():.4f}')

    # 4. Test Loop
    print("Evaluating Model...")
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out[data.test_mask].argmax(dim=1)
        correct = int(pred.eq(data.y[data.test_mask]).sum().item())
        total = int(data.test_mask.sum().item())
            
    acc = correct / total
    print(f'Final Full-Dataset Testing Accuracy: {acc:.4f}')

    # 5. Save the Model
    torch.save(model.state_dict(), 'yelp_gat_full_model.pt')
    print("Model successfully saved to 'yelp_gat_full_model.pt'")


Loading full dataset... (Requires 16GB+ System RAM)
Loaded 1987897 users. Building network topology...
Scaling features...
Graph Built: 1987897 nodes, 14611748 edges.
Using device: mps
Beginning Full-Batch Training...
Epoch  10 | Loss: 3.1336
Epoch  20 | Loss: 0.8774
Epoch  30 | Loss: 0.5951
Epoch  40 | Loss: 0.5607
Epoch  50 | Loss: 0.4968
Epoch  60 | Loss: 0.4220
Epoch  70 | Loss: 0.4040
Epoch  80 | Loss: 0.3694
Epoch  90 | Loss: 0.3823
Epoch 100 | Loss: 0.3310
Epoch 110 | Loss: 0.3148
Epoch 120 | Loss: 0.2973
Epoch 130 | Loss: 0.3007
Epoch 140 | Loss: 0.2942
Epoch 150 | Loss: 0.2845
Epoch 160 | Loss: 0.2545
Epoch 170 | Loss: 0.2471
Epoch 180 | Loss: 0.2367
Epoch 190 | Loss: 0.2307
Epoch 200 | Loss: 0.2325
Evaluating Model...
Final Full-Dataset Testing Accuracy: 0.9585
Model successfully saved to 'yelp_gat_full_model.pt'
